# Last.fm Tag Enrichment

This notebook enriches each unique artist–track combination with genre tags retrieved from the Last.fm API.

## 1. Load libraries

In [ ]:
import pandas as pd
import time
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.clients.lastfm_client import LastFMClient
from src.config import LASTFM_API_KEY

## 2. Load data from last.fm

In [ ]:
DATA_PATH = Path("../data/processed/lastfm_scrobbles_clean.parquet")

In [ ]:
df = pd.read_parquet(DATA_PATH)

unique_artists = df["artist"].dropna().unique()

## 3. Connect to Last.fm API

In [ ]:
client = LastFMClient(api_key=LASTFM_API_KEY)

## 4. Retrieve genre tags

In [ ]:
artist_tags_cache = {}

for i, artist in enumerate(unique_artists):
    
    if artist in artist_tags_cache:
        continue
    
    tags = client.get_artist_tags(artist)
    artist_tags_cache[artist] = tags
    
    if i % 50 == 0:
        print(i)
    
    time.sleep(0.3)

In [ ]:
df["tags"] = df["artist"].map(artist_tags_cache)

## 5. Remove duplicate artist–track combinations

In [ ]:
model_df = df.drop_duplicates(subset=["artist_clean", "track_clean"])

In [ ]:
model_df.shape

In [ ]:
df[["artist_clean", "track_clean"]].duplicated().sum()

In [ ]:
df["artist_clean"].nunique()

In [ ]:
df["track_clean"].nunique()

In [ ]:
df["artist_clean"].value_counts().head(20)

In [ ]:
df.groupby("artist_clean")["track_clean"].nunique().sort_values(ascending=False).head(20)

## 6. Initial cleaning of tags

In [ ]:
def clean_tags(tags):
    if not isinstance(tags, list):
        return []
    
    return list(set([t.lower() for t in tags if len(t) > 2 and "seen live" not in t]))

In [ ]:
model_df["tags_clean"] = model_df["tags"].apply(clean_tags)

## 7. Sanity check

In [ ]:
model_df["tags_clean"].explode().value_counts().head(20)

## 8. Export enriched dataset

In [ ]:
model_df.to_parquet("../data/processed/lastfm_scrobbles_with_tags.parquet", index=False)

### Output

`data/processed/lastfm_scrobbles_with_tags.parquet`

Used in **04_lastfm_tag_normnalization.ipynb**.